# Global PCA and K-means：K⁺／Na⁺ 

這份 Notebook 將 10 個系統放入同一個全域特徵空間：

1. 每個系統載入一條 structural-property time series 與一條 dihedral-angle time series。
2. 兩條 time series 等距重採樣至共同長度。
3. 合併全部系統後，只做一次 global standardization。
4. 只建立一次 global PCA。
5. 在標準化特徵上執行一次 global K-means。
6. 保存各系統 PCA centroid 與 cluster population。

K-means 預設使用 4 群。這是可調整的研究設定，不代表程式已自動證明存在四種結合模式；群集仍需搭配代表構型與接觸分析解讀。


## 1. 環境準備

需要 Python 3、NumPy、Matplotlib 與 scikit-learn。

    pip install numpy matplotlib scikit-learn


In [ ]:
# ============================================================
# 2. 載入套件
# ============================================================

from pathlib import Path
import csv

import matplotlib.pyplot as plt
import numpy as np
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

print(f"NumPy: {np.__version__}")


## 3. 全域設定

原始分析將二面角視為一般線性數值，但 −89° 與 +89° 在 180° 週期下其實非常接近。新版預設將角度轉換為 sine／cosine 週期特徵，再與 structural property 一起送入 PCA。

若輸入角度不是 180° 週期，請修改 DIHEDRAL_PERIOD_DEG。


In [ ]:
# ============================================================
# 4. 全域設定
# ============================================================

DIHEDRAL_FILES = {
    "K": Path(
        "/dicos_ui_home/chrysaliso/code_python_analysis/"
        "3rep_analysis/dihedral_analysis.npz"
    ),
    "Na": Path(
        "/dicos_ui_home/chrysaliso/code_python_analysis/"
        "3rep_analysis/dihedral_analysis_na.npz"
    ),
}

# folded torsion 位於 [-90, 90)，因此週期為 180°。
DIHEDRAL_PERIOD_DEG = 180.0
USE_CIRCULAR_DIHEDRAL = True

N_CLUSTERS = 4
RANDOM_STATE = 42
KMEANS_N_INIT = 20

# 只影響繪圖抽樣，不影響 PCA 或 K-means 計算。
PLOT_MAX_POINTS_PER_CLUSTER = 5000

OUTPUT_DIR = Path("./results/global_pca_kmeans")

CLUSTER_COLORS = [
    "#E64B35",
    "#4DBBD5",
    "#00A087",
    "#3C5488",
]

if N_CLUSTERS > len(CLUSTER_COLORS):
    raise ValueError(
        "N_CLUSTERS 超過 CLUSTER_COLORS 數量，請增加顏色。"
    )

print(f"Number of clusters: {N_CLUSTERS}")
print(f"Circular dihedral encoding: {USE_CIRCULAR_DIHEDRAL}")
print(f"Output directory: {OUTPUT_DIR.resolve()}")


## 5. 系統設定

每個系統需要：

- system_id：NPZ 與 CSV 使用的唯一名稱
- display_label：圖片顯示名稱
- ion_type：K 或 Na
- docking_score：僅作 metadata，不直接當成 PCA feature
- dihedral_key：從對應 dihedral NPZ 讀取的 key
- property_file：另一個逐 frame 數值的 NPZ
- property_key：若知道 key 可直接填入；若設為 None，程式要求檔案只有一個可用的一維數值陣列

注意：檔名包含 docking score 不代表 docking score 本身是一條 time series。請確認 property_file 裡的數值實際代表哪一種物理量。


In [ ]:
# ============================================================
# 6. 十個系統的輸入設定
# ============================================================

SYSTEMS = [
    {
        "system_id": "K_606",
        "display_label": "K+ 6.06",
        "ion_type": "K",
        "docking_score": "6.06",
        "dihedral_key": "sys_0_mean",
        "property_file": Path(
            "/dicos_ui_home/chrysaliso/code_python_analysis/"
            "3rep_analysis/analysis_data/"
            "data_Docking_score__606_K.npz"
        ),
        "property_key": None,
    },
    {
        "system_id": "K_687",
        "display_label": "K+ 6.87",
        "ion_type": "K",
        "docking_score": "6.87",
        "dihedral_key": "sys_1_mean",
        "property_file": Path(
            "/dicos_ui_home/chrysaliso/code_python_analysis/"
            "3rep_analysis/analysis_data/"
            "data_Docking_score__687_K.npz"
        ),
        "property_key": None,
    },
    {
        "system_id": "K_688",
        "display_label": "K+ 6.88",
        "ion_type": "K",
        "docking_score": "6.88",
        "dihedral_key": "sys_2_mean",
        "property_file": Path(
            "/dicos_ui_home/chrysaliso/code_python_analysis/"
            "3rep_analysis/analysis_data/"
            "data_Docking_score__688_K.npz"
        ),
        "property_key": None,
    },
    {
        "system_id": "K_688_2",
        "display_label": "K+ 6.88_2",
        "ion_type": "K",
        "docking_score": "6.88_2",
        "dihedral_key": "sys_3_mean",
        "property_file": Path(
            "/dicos_ui_home/chrysaliso/code_python_analysis/"
            "3rep_analysis/analysis_data/"
            "data_Docking_score__688_2_K.npz"
        ),
        "property_key": None,
    },
    {
        "system_id": "K_721",
        "display_label": "K+ 7.21",
        "ion_type": "K",
        "docking_score": "7.21",
        "dihedral_key": "sys_4_mean",
        "property_file": Path(
            "/dicos_ui_home/chrysaliso/code_python_analysis/"
            "3rep_analysis/analysis_data/"
            "data_Docking_score__721_K.npz"
        ),
        "property_key": None,
    },
    {
        "system_id": "Na_606",
        "display_label": "Na+ 6.06",
        "ion_type": "Na",
        "docking_score": "6.06",
        "dihedral_key": "sys_0_mean",
        "property_file": Path(
            "/dicos_ui_home/chrysaliso/code_python_analysis/"
            "3rep_analysis/analysis_data/"
            "data_Docking_score__606_na.npz"
        ),
        "property_key": None,
    },
    {
        "system_id": "Na_687",
        "display_label": "Na+ 6.87",
        "ion_type": "Na",
        "docking_score": "6.87",
        "dihedral_key": "sys_1_mean",
        "property_file": Path(
            "/dicos_ui_home/chrysaliso/code_python_analysis/"
            "3rep_analysis/analysis_data/"
            "data_Docking_score__687_na.npz"
        ),
        "property_key": None,
    },
    {
        "system_id": "Na_688",
        "display_label": "Na+ 6.88",
        "ion_type": "Na",
        "docking_score": "6.88",
        "dihedral_key": "sys_2_mean",
        "property_file": Path(
            "/dicos_ui_home/chrysaliso/code_python_analysis/"
            "3rep_analysis/analysis_data/"
            "data_Docking_score__688_na.npz"
        ),
        "property_key": None,
    },
    {
        "system_id": "Na_688_2",
        "display_label": "Na+ 6.88_2",
        "ion_type": "Na",
        "docking_score": "6.88_2",
        "dihedral_key": "sys_3_mean",
        "property_file": Path(
            "/dicos_ui_home/chrysaliso/code_python_analysis/"
            "3rep_analysis/analysis_data/"
            "data_Docking_score__688_2_na.npz"
        ),
        "property_key": None,
    },
    {
        "system_id": "Na_721",
        "display_label": "Na+ 7.21",
        "ion_type": "Na",
        "docking_score": "7.21",
        "dihedral_key": "sys_4_mean",
        "property_file": Path(
            "/dicos_ui_home/chrysaliso/code_python_analysis/"
            "3rep_analysis/analysis_data/"
            "data_Docking_score__721_na.npz"
        ),
        "property_key": None,
    },
]

system_ids = [system["system_id"] for system in SYSTEMS]

if len(system_ids) != len(set(system_ids)):
    raise ValueError("SYSTEMS 中的 system_id 必須唯一。")

print(f"Configured systems: {len(SYSTEMS)}")


## 7. 資料載入與重採樣函數

新版不使用固定步長切片，而是以 np.linspace 產生從第一點到最後一點的等距索引，避免兩條 time series 長度不是整數倍時只取到前半段。

所有輸入都會轉為一維有限數值陣列。


In [ ]:
# ============================================================
# 8. 輸入處理函數
# ============================================================


def require_file(file_path, description):
    """確認輸入檔存在。"""
    if not file_path.is_file():
        raise FileNotFoundError(
            f"找不到{description}：{file_path}"
        )


def to_numeric_1d(array, description):
    """將 Nx1 或一維數值轉成有限的一維 float array。"""
    values = np.asarray(array)
    values = np.squeeze(values)

    if values.ndim != 1:
        raise ValueError(
            f"{description} 必須是一維或 Nx1，目前 shape={values.shape}"
        )

    if values.size < 2:
        raise ValueError(f"{description} 至少需要兩個數值。")

    try:
        values = values.astype(float)
    except (TypeError, ValueError) as error:
        raise ValueError(
            f"{description} 無法轉成 float。"
        ) from error

    if not np.all(np.isfinite(values)):
        raise ValueError(f"{description} 包含 NaN 或 infinity。")

    return values


def load_npz_series(file_path, requested_key, description):
    """
    從 NPZ 載入一條 time series。

    requested_key 為 None 時，只接受恰好一個可轉成一維數值的 key。
    """
    require_file(file_path, description)

    with np.load(file_path, allow_pickle=False) as data:
        if requested_key is not None:
            if requested_key not in data.files:
                raise KeyError(
                    f"{description} 缺少 key '{requested_key}'。"
                    f"可用 keys：{data.files}"
                )
            return to_numeric_1d(
                data[requested_key],
                f"{description}:{requested_key}",
            )

        candidates = []

        for key in data.files:
            try:
                values = to_numeric_1d(
                    data[key],
                    f"{description}:{key}",
                )
            except (TypeError, ValueError):
                continue

            candidates.append((key, values))

        if len(candidates) != 1:
            candidate_keys = [key for key, _ in candidates]
            raise ValueError(
                f"{description} 找到 {len(candidates)} 個可用的一維數值 keys："
                f"{candidate_keys}。請在 SYSTEMS 中明確設定 property_key。"
            )

        selected_key, selected_values = candidates
        print(f"  Auto-selected property key: {selected_key}")
        return selected_values


def resample_evenly(values, target_length):
    """以包含首尾的等距索引重採樣至指定長度。"""
    if target_length > len(values):
        raise ValueError("target_length 不可大於原始陣列長度。")

    indices = np.linspace(
        0,
        len(values) - 1,
        num=target_length,
    )
    indices = np.rint(indices).astype(int)

    return values[indices], indices


def build_features(property_values, dihedral_deg):
    """
    建立 PCA／K-means 使用的特徵。

    週期模式：property、sin(angle)、cos(angle)
    線性模式：property、angle
    """
    if USE_CIRCULAR_DIHEDRAL:
        phase_rad = (
            2.0
            * np.pi
            * dihedral_deg
            / DIHEDRAL_PERIOD_DEG
        )
        features = np.column_stack([
            property_values,
            np.sin(phase_rad),
            np.cos(phase_rad),
        ])
        feature_names = np.asarray([
            "property",
            "dihedral_sin",
            "dihedral_cos",
        ])
    else:
        features = np.column_stack([
            property_values,
            dihedral_deg,
        ])
        feature_names = np.asarray([
            "property",
            "dihedral_degree",
        ])

    return features, feature_names


## 9. 載入十個系統並建立全域矩陣

每個系統的兩條特徵先在系統內對齊長度，再將十個系統垂直合併。PCA 與 K-means 不會分別對每個系統重算。


In [ ]:
# ============================================================
# 10. 建立 global feature matrix
# ============================================================

for ion_type, dihedral_file in DIHEDRAL_FILES.items():
    require_file(
        dihedral_file,
        f" {ion_type} dihedral NPZ",
    )

all_feature_blocks = []
all_property_blocks = []
all_dihedral_blocks = []
point_system_index_blocks = []
point_frame_index_blocks = []
property_source_index_blocks = []
dihedral_source_index_blocks = []
system_point_counts = []
feature_names = None

dihedral_handles = {
    ion_type: np.load(file_path, allow_pickle=False)
    for ion_type, file_path in DIHEDRAL_FILES.items()
}

try:
    for system_index, system in enumerate(SYSTEMS):
        print(
            f"[{system_index + 1}/{len(SYSTEMS)}] "
            f"{system['display_label']}"
        )

        ion_type = system["ion_type"]

        if ion_type not in dihedral_handles:
            raise KeyError(
                f"找不到 ion_type={ion_type} 的 dihedral NPZ。"
            )

        dihedral_data = dihedral_handles[ion_type]
        dihedral_key = system["dihedral_key"]

        if dihedral_key not in dihedral_data.files:
            raise KeyError(
                f"{DIHEDRAL_FILES[ion_type]} 缺少 key "
                f"'{dihedral_key}'。可用 keys：{dihedral_data.files}"
            )

        dihedral_values = to_numeric_1d(
            dihedral_data[dihedral_key],
            f"{system['system_id']} dihedral",
        )

        property_values = load_npz_series(
            system["property_file"],
            system["property_key"],
            f"{system['system_id']} property NPZ",
        )

        target_length = min(
            len(property_values),
            len(dihedral_values),
        )

        property_aligned, property_indices = resample_evenly(
            property_values,
            target_length,
        )
        dihedral_aligned, dihedral_indices = resample_evenly(
            dihedral_values,
            target_length,
        )

        feature_block, current_feature_names = build_features(
            property_aligned,
            dihedral_aligned,
        )

        if feature_names is None:
            feature_names = current_feature_names
        elif not np.array_equal(
            feature_names,
            current_feature_names,
        ):
            raise RuntimeError("各系統的 feature names 不一致。")

        all_feature_blocks.append(feature_block)
        all_property_blocks.append(property_aligned)
        all_dihedral_blocks.append(dihedral_aligned)
        point_system_index_blocks.append(
            np.full(target_length, system_index, dtype=int)
        )
        point_frame_index_blocks.append(
            np.arange(target_length, dtype=int)
        )
        property_source_index_blocks.append(property_indices)
        dihedral_source_index_blocks.append(dihedral_indices)
        system_point_counts.append(target_length)

        print(
            f"  property={len(property_values):,}, "
            f"dihedral={len(dihedral_values):,}, "
            f"used={target_length:,}"
        )

finally:
    for handle in dihedral_handles.values():
        handle.close()

X_raw = np.vstack(all_feature_blocks)
property_total = np.concatenate(all_property_blocks)
dihedral_total = np.concatenate(all_dihedral_blocks)
point_system_index = np.concatenate(
    point_system_index_blocks
)
point_frame_index = np.concatenate(
    point_frame_index_blocks
)
property_source_index = np.concatenate(
    property_source_index_blocks
)
dihedral_source_index = np.concatenate(
    dihedral_source_index_blocks
)
system_point_counts = np.asarray(
    system_point_counts,
    dtype=int,
)

print(f"Global feature matrix: {X_raw.shape}")
print(f"Feature names: {feature_names.tolist()}")


## 11. Global standardization、PCA 與 K-means

StandardScaler、PCA 與 K-means 都只對合併後的全域矩陣 fit 一次。K-means 使用標準化特徵；因 PCA 保留全部需要的二維顯示資訊，cluster centers 再轉換到 PCA 座標繪圖。


In [ ]:
# ============================================================
# 12. 執行 PCA 與 K-means
# ============================================================

if N_CLUSTERS < 2:
    raise ValueError("N_CLUSTERS 至少為 2。")

if N_CLUSTERS >= len(X_raw):
    raise ValueError("N_CLUSTERS 必須小於總資料點數。")

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_raw)

pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)
explained_variance_ratio = pca.explained_variance_ratio_

kmeans = KMeans(
    n_clusters=N_CLUSTERS,
    random_state=RANDOM_STATE,
    n_init=KMEANS_N_INIT,
)
cluster_labels = kmeans.fit_predict(X_scaled)
cluster_centers_pca = pca.transform(
    kmeans.cluster_centers_
)

system_centroids_pca = np.zeros(
    (len(SYSTEMS), 2),
    dtype=float,
)
cluster_populations = np.zeros(
    (len(SYSTEMS), N_CLUSTERS),
    dtype=float,
)

for system_index in range(len(SYSTEMS)):
    system_mask = point_system_index == system_index
    system_centroids_pca[system_index] = np.mean(
        X_pca[system_mask],
        axis=0,
    )

    counts = np.bincount(
        cluster_labels[system_mask],
        minlength=N_CLUSTERS,
    )
    cluster_populations[system_index] = (
        counts / counts.sum()
    )

if not np.allclose(
    np.sum(cluster_populations, axis=1),
    1.0,
):
    raise RuntimeError(
        "部分系統的 cluster populations 加總不為 1。"
    )

print(
    f"PC1: {explained_variance_ratio[0]:.2%}, "
    f"PC2: {explained_variance_ratio[1]:.2%}"
)
print(f"K-means inertia: {kmeans.inertia_:.6f}")


## 13. 儲存 NPZ、cluster population CSV 與摘要

NPZ 保存每個 PCA 點與 cluster label；CSV 只保存系統層級的 centroid 和 cluster population，避免產生過大的逐點 CSV。


In [ ]:
# ============================================================
# 14. 儲存計算結果
# ============================================================

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

system_id_array = np.asarray([
    system["system_id"]
    for system in SYSTEMS
])
display_label_array = np.asarray([
    system["display_label"]
    for system in SYSTEMS
])
ion_type_array = np.asarray([
    system["ion_type"]
    for system in SYSTEMS
])
docking_score_array = np.asarray([
    system["docking_score"]
    for system in SYSTEMS
])

npz_path = OUTPUT_DIR / "global_pca_kmeans_results.npz"
np.savez_compressed(
    npz_path,
    X_raw=X_raw,
    X_scaled=X_scaled,
    X_pca=X_pca,
    property_values=property_total,
    dihedral_deg=dihedral_total,
    feature_names=feature_names,
    point_system_index=point_system_index,
    point_frame_index=point_frame_index,
    property_source_index=property_source_index,
    dihedral_source_index=dihedral_source_index,
    cluster_labels=cluster_labels,
    cluster_centers_pca=cluster_centers_pca,
    cluster_populations=cluster_populations,
    system_centroids_pca=system_centroids_pca,
    system_ids=system_id_array,
    display_labels=display_label_array,
    ion_types=ion_type_array,
    docking_scores=docking_score_array,
    system_point_counts=system_point_counts,
    explained_variance_ratio=explained_variance_ratio,
    scaler_mean=scaler.mean_,
    scaler_scale=scaler.scale_,
    pca_components=pca.components_,
    kmeans_inertia=np.asarray(kmeans.inertia_),
    n_clusters=np.asarray(N_CLUSTERS),
    random_state=np.asarray(RANDOM_STATE),
    dihedral_period_deg=np.asarray(
        DIHEDRAL_PERIOD_DEG
    ),
    circular_dihedral=np.asarray(
        USE_CIRCULAR_DIHEDRAL
    ),
)

csv_path = OUTPUT_DIR / "system_cluster_summary.csv"
with csv_path.open("w", newline="", encoding="utf-8") as csv_file:
    writer = csv.writer(csv_file)

    header = [
        "system_id",
        "display_label",
        "ion_type",
        "docking_score",
        "point_count",
        "centroid_pc1",
        "centroid_pc2",
    ] + [
        f"cluster_{cluster_index + 1}_population"
        for cluster_index in range(N_CLUSTERS)
    ]

    writer.writerow(header)

    for system_index, system in enumerate(SYSTEMS):
        writer.writerow([
            system["system_id"],
            system["display_label"],
            system["ion_type"],
            system["docking_score"],
            int(system_point_counts[system_index]),
            f"{system_centroids_pca[system_index, 0]:.6f}",
            f"{system_centroids_pca[system_index, 1]:.6f}",
            *[
                f"{population:.6f}"
                for population in cluster_populations[
                    system_index
                ]
            ],
        ])

summary_path = OUTPUT_DIR / "run_summary.txt"
with summary_path.open("w", encoding="utf-8") as summary_file:
    summary_file.write("Global PCA and K-means\n")
    summary_file.write(f"Systems: {len(SYSTEMS)}\n")
    summary_file.write(f"Total points: {len(X_raw)}\n")
    summary_file.write(
        f"Feature names: {feature_names.tolist()}\n"
    )
    summary_file.write(
        f"Circular dihedral: {USE_CIRCULAR_DIHEDRAL}\n"
    )
    summary_file.write(
        f"Dihedral period: {DIHEDRAL_PERIOD_DEG} degree\n"
    )
    summary_file.write(
        f"PC1 variance: {explained_variance_ratio[0]:.8f}\n"
    )
    summary_file.write(
        f"PC2 variance: {explained_variance_ratio[1]:.8f}\n"
    )
    summary_file.write(f"Clusters: {N_CLUSTERS}\n")
    summary_file.write(
        f"Random state: {RANDOM_STATE}\n"
    )
    summary_file.write(
        f"K-means inertia: {kmeans.inertia_:.8f}\n"
    )

print(f"NPZ saved: {npz_path.resolve()}")
print(f"CSV saved: {csv_path.resolve()}")
print(f"Summary saved: {summary_path.resolve()}")


## 15. 從 NPZ 載入並繪圖

左圖顯示 global PCA 中的 K-means clusters；大 X 為 cluster centers。為避免資料量過大，散點只做等量抽樣顯示，但中心與 population 使用全部資料計算。

右圖顯示每個系統屬於各 cluster 的比例。


In [ ]:
# ============================================================
# 16. 載入繪圖數據
# ============================================================

npz_path = OUTPUT_DIR / "global_pca_kmeans_results.npz"

if not npz_path.is_file():
    raise FileNotFoundError(
        f"找不到 NPZ：{npz_path}\n"
        "請先執行 PCA／K-means 計算 cell。"
    )

data = np.load(npz_path, allow_pickle=False)

required_keys = {
    "X_pca",
    "cluster_labels",
    "cluster_centers_pca",
    "cluster_populations",
    "display_labels",
    "explained_variance_ratio",
    "n_clusters",
}

missing_keys = required_keys.difference(data.files)

if missing_keys:
    raise KeyError(f"NPZ 缺少欄位：{sorted(missing_keys)}")

plot_X_pca = data["X_pca"]
plot_cluster_labels = data["cluster_labels"]
plot_cluster_centers = data["cluster_centers_pca"]
plot_cluster_populations = data["cluster_populations"]
plot_display_labels = data["display_labels"].astype(str)
plot_variance = data["explained_variance_ratio"]
plot_n_clusters = int(data["n_clusters"])

print(f"Loaded: {npz_path.resolve()}")
print(f"Points: {len(plot_X_pca):,}")


In [ ]:
# ============================================================
# 17. PCA clusters 與 cluster populations
# ============================================================

fig, axes = plt.subplots(
    1,
    2,
    figsize=(24, 9),
    dpi=300,
)

ax_pca, ax_bar = axes
rng = np.random.default_rng(RANDOM_STATE)

for cluster_index in range(plot_n_clusters):
    cluster_point_indices = np.where(
        plot_cluster_labels == cluster_index
    )[0]

    if len(cluster_point_indices) > PLOT_MAX_POINTS_PER_CLUSTER:
        display_indices = rng.choice(
            cluster_point_indices,
            size=PLOT_MAX_POINTS_PER_CLUSTER,
            replace=False,
        )
    else:
        display_indices = cluster_point_indices

    ax_pca.scatter(
        plot_X_pca[display_indices, 0],
        plot_X_pca[display_indices, 1],
        s=10,
        alpha=0.25,
        color=CLUSTER_COLORS[cluster_index],
        linewidths=0,
        label=f"Cluster {cluster_index + 1}",
    )

ax_pca.scatter(
    plot_cluster_centers[:, 0],
    plot_cluster_centers[:, 1],
    s=280,
    marker="X",
    color=CLUSTER_COLORS[:plot_n_clusters],
    edgecolors="black",
    linewidths=1.5,
    label="Cluster centers",
    zorder=5,
)

ax_pca.set_title(
    "Global PCA with K-means",
    fontsize=30,
)
ax_pca.set_xlabel(
    f"PC1 ({plot_variance[0] * 100:.1f}%)",
    fontsize=30,
)
ax_pca.set_ylabel(
    f"PC2 ({plot_variance[1] * 100:.1f}%)",
    fontsize=30,
)
ax_pca.tick_params(
    axis="both",
    which="major",
    labelsize=24,
)
ax_pca.grid(
    linestyle="--",
    alpha=0.35,
)
ax_pca.legend(
    fontsize=15,
    frameon=True,
    framealpha=0.9,
)

x_positions = np.arange(len(plot_display_labels))
bar_bottom = np.zeros(len(plot_display_labels))

for cluster_index in range(plot_n_clusters):
    populations = plot_cluster_populations[:, cluster_index]

    ax_bar.bar(
        x_positions,
        populations,
        bottom=bar_bottom,
        color=CLUSTER_COLORS[cluster_index],
        width=0.78,
        label=f"Cluster {cluster_index + 1}",
    )

    bar_bottom += populations

ax_bar.set_title(
    "Cluster Populations by System",
    fontsize=30,
)
ax_bar.set_xlabel(
    "System",
    fontsize=30,
)
ax_bar.set_ylabel(
    "Population",
    fontsize=30,
)
ax_bar.set_ylim(0, 1)
ax_bar.set_xticks(x_positions)
ax_bar.set_xticklabels(
    plot_display_labels,
    rotation=60,
    ha="right",
)
ax_bar.tick_params(
    axis="y",
    which="major",
    labelsize=24,
)
ax_bar.tick_params(
    axis="x",
    which="major",
    labelsize=16,
)
ax_bar.grid(
    axis="y",
    linestyle="--",
    alpha=0.35,
)
ax_bar.legend(
    fontsize=15,
    frameon=True,
    framealpha=0.9,
)

fig.tight_layout()

png_path = OUTPUT_DIR / "figure_global_pca_kmeans.png"
pdf_path = OUTPUT_DIR / "figure_global_pca_kmeans.pdf"

fig.savefig(
    png_path,
    dpi=300,
    bbox_inches="tight",
)
fig.savefig(
    pdf_path,
    bbox_inches="tight",
)

plt.show()

print(f"PNG saved: {png_path.resolve()}")
print(f"PDF saved: {pdf_path.resolve()}")


## 18. 結果判讀與檢查

完成後確認：

- 十個系統都成功載入，沒有缺少 property key 或 dihedral key。
- PCA 與 K-means 都只 fit 一次全域矩陣。
- Cluster populations 每個系統加總為 1。
- 固定 RANDOM_STATE 後，重跑應得到相同結果。
- 二面角若為 −90° 到 90° folded angle，DIHEDRAL_PERIOD_DEG 使用 180。
- K-means 群集只是數據分區；應再由每群代表 frame、接觸、COM distance 或能量確認其結構意義。

若 property_file 實際儲存的是 COM distance、RMSD 或其他特徵，建議將 feature_names 中的 property 改成真實物理名稱，避免論文中誤稱為 docking score。
